In [2]:
import numpy as np
import pandas as pd
import json

# ==============================================================================
# 1. MOIL MINES MASTER CONFIGURATION (From your team's IBM & MCDR research)
# ==============================================================================
MOIL_MINES = [
    {
        "mine_id": "40MPR01002",
        "name": "Balaghat (Bharweli)",
        "state": "Madhya Pradesh",
        "lat": 21.8415, "lon": 80.2185,
        "type": "Underground",
        "annual_capacity": 500000,
        "avg_grade_mn": 42.5,
        "depth_m": 435,
        "host_rock": "Gondite/Schist",
        "water_table_mrl": 45
    },
    {
        "mine_id": "40MSH05002",
        "name": "Dongri Buzurg",
        "state": "Maharashtra",
        "lat": 21.5490, "lon": 79.6830,
        "type": "Opencast",
        "annual_capacity": 300000,
        "avg_grade_mn": 46.5,  # Dioxide grade
        "depth_m": 160,
        "host_rock": "Gondite/Schist-Gneiss",
        "water_table_mrl": 50
    },
    {
        "mine_id": "40MSH05001",
        "name": "Chikla",
        "state": "Maharashtra",
        "lat": 21.5371, "lon": 79.7485,
        "type": "Underground",
        "annual_capacity": 180000,
        "avg_grade_mn": 39.5,
        "depth_m": 169,
        "host_rock": "Mica Schist",
        "water_table_mrl": 40
    },
    {
        "mine_id": "40MSH14004",
        "name": "Gumgaon",
        "state": "Maharashtra",
        "lat": 21.6667, "lon": 78.9833,
        "type": "Underground",
        "annual_capacity": 305200,
        "avg_grade_mn": 38.0,
        "depth_m": 162,
        "host_rock": "Mica Schist/Quartzite",
        "water_table_mrl": 35
    },
    {
        "mine_id": "40MSH14008",
        "name": "Kandri",
        "state": "Maharashtra",
        "lat": 21.3928, "lon": 79.2722,
        "type": "Underground",
        "annual_capacity": 63000,
        "avg_grade_mn": 40.0,
        "depth_m": 137,
        "host_rock": "Gondite",
        "water_table_mrl": 42
    },
    {
        "mine_id": "40MSH14010",
        "name": "Munsar (Mansar)",
        "state": "Maharashtra",
        "lat": 21.4040, "lon": 79.2830,
        "type": "Underground",
        "annual_capacity": 89992,
        "avg_grade_mn": 37.5,
        "depth_m": 190,
        "host_rock": "Mica Schist",
        "water_table_mrl": 48
    },
    {
        "mine_id": "40MPR01016",
        "name": "Sitapatore",
        "state": "Madhya Pradesh",
        "lat": 21.6667, "lon": 79.6667,
        "type": "Opencast",
        "annual_capacity": 17000,
        "avg_grade_mn": 35.0,
        "depth_m": 67,
        "host_rock": "Quartzite/Schist",
        "water_table_mrl": 63
    },
    {
        "mine_id": "40MSH14002",
        "name": "Beldongri",
        "state": "Maharashtra",
        "lat": 21.3900, "lon": 79.3000,
        "type": "Underground",
        "annual_capacity": 40000,
        "avg_grade_mn": 34.0,
        "depth_m": 122,
        "host_rock": "Mica Schist",
        "water_table_mrl": 38
    },
    {
        "mine_id": "40MPR01018",
        "name": "Ukwa",
        "state": "Madhya Pradesh",
        "lat": 21.9682, "lon": 80.4687,
        "type": "Underground",
        "annual_capacity": 160000,
        "avg_grade_mn": 37.0,
        "depth_m": 250,
        "host_rock": "Sericite Schist",
        "water_table_mrl": 45
    },
    {
        "mine_id": "40MPR01069",
        "name": "Tirodi",
        "state": "Madhya Pradesh",
        "lat": 21.6841, "lon": 79.7125,
        "type": "Opencast",
        "annual_capacity": 140000,
        "avg_grade_mn": 34.5,
        "depth_m": 90,
        "host_rock": "Quartzite",
        "water_table_mrl": 52
    }
]

# Save master metadata JSON
with open('moil_mines_master.json', 'w') as f:
    json.dump(MOIL_MINES, f, indent=2)
print("Saved moil_mines_master.json!")

# ==============================================================================
# 2. GENERATE DATASET 1: MANGANESE PROSPECTIVITY (For Hardik's Model 1)
# ==============================================================================
# Generates 500 spatial sample points across the Nagpur-Balaghat belt:
# - Positive samples: near known MOIL mine corridors
# - Negative samples: background / barren areas further away
np.random.seed(42)

prospectivity_rows = []
for mine in MOIL_MINES:
    # 25 positive / mineralization cluster samples around each mine (within 0.05 deg ~ 5km)
    for _ in range(25):
        lat = mine['lat'] + np.random.normal(0, 0.025)
        lon = mine['lon'] + np.random.normal(0, 0.025)
        dist_km = np.sqrt(((lat - mine['lat'])*111)**2 + ((lon - mine['lon'])*103)**2)
        
        # Manganese alteration signatures: higher SWIR2/B12 absorption, distinct ferrous ratio
        b02 = np.clip(np.random.normal(0.066, 0.018), 0.03, 0.11)
        b03 = np.clip(np.random.normal(0.095, 0.022), 0.04, 0.14)
        b04 = np.clip(np.random.normal(0.135, 0.028), 0.06, 0.20)
        b08 = np.clip(np.random.normal(0.31, 0.06), 0.15, 0.45)
        b11 = np.clip(np.random.normal(0.30, 0.055), 0.10, 0.45)
        b12 = np.clip(np.random.normal(0.25, 0.045), 0.10, 0.35)
        
        ndvi = (b08 - b04) / (b08 + b04)
        ferrous_index = b11 / (b08 + 1e-6)
        clay_alteration = b11 / (b12 + 1e-6)
        elevation = np.clip(np.random.normal(265, 55), 2, 480) # Satpura foothills
        slope = np.clip(np.random.normal(11,5.5), 0.5, 20)
        
        # Grade: bounded by MOIL's real bands (25% to 48%)
        mn_grade = np.clip(mine['avg_grade_mn'] + np.random.normal(0, 2.5), 22.0, 48.0)
        
        prospectivity_rows.append({
            "latitude": round(lat, 5),
            "longitude": round(lon, 5),
            "nearest_mine_id": mine["mine_id"],
            "distance_to_deposit_km": round(dist_km, 2),
            "b02": round(b02, 4), "b03": round(b03, 4), "b04": round(b04, 4),
            "b08": round(b08, 4), "b11": round(b11, 4), "b12": round(b12, 4),
            "ndvi": round(ndvi, 4),
            "ferrous_index": round(ferrous_index, 4),
            "clay_alteration": round(clay_alteration, 4),
            "elevation_m": round(elevation, 1),
            "slope_deg": round(slope, 1),
            "mn_grade_pct": round(mn_grade, 2),
            "target_occurrence": 1  # Confirmed Manganese Occurrence
        })

# 250 Negative / background points across Central India (barren / no Mn mineralization)
for _ in range(250):
    lat = np.random.uniform(21.1, 22.2)
    lon = np.random.uniform(78.8, 80.7)
    
    # Calculate min distance to any mine
    min_dist = min([np.sqrt(((lat - m['lat'])*111)**2 + ((lon - m['lon'])*103)**2) for m in MOIL_MINES])
    
    # Only keep as negative if > 8km away from known deposits
    if min_dist > 8.0:
        b02 = np.clip(np.random.normal(0.074, 0.020), 0.03, 0.13)
        b03 = np.clip(np.random.normal(0.105, 0.024), 0.04, 0.16)
        b04 = np.clip(np.random.normal(0.150, 0.030), 0.06, 0.22)
        b08 = np.clip(np.random.normal(0.33, 0.07), 0.15, 0.48)
        b11 = np.clip(np.random.normal(0.23, 0.055), 0.08, 0.40)
        b12 = np.clip(np.random.normal(0.20, 0.045), 0.08, 0.32)
        
        ndvi = (b08 - b04) / (b08 + b04)
        ferrous_index = b11 / (b08 + 1e-6)
        clay_alteration = b11 / (b12 + 1e-6)
        elevation = np.clip(np.random.normal(310, 55), 200, 480)
        slope = np.clip(np.random.normal(6.5, 5.5), 0.5, 20)
        
        prospectivity_rows.append({
            "latitude": round(lat, 5),
            "longitude": round(lon, 5),
            "nearest_mine_id": "NONE",
            "distance_to_deposit_km": round(min_dist, 2),
            "b02": round(b02, 4), "b03": round(b03, 4), "b04": round(b04, 4),
            "b08": round(b08, 4), "b11": round(b11, 4), "b12": round(b12, 4),
            "ndvi": round(ndvi, 4),
            "ferrous_index": round(ferrous_index, 4),
            "clay_alteration": round(clay_alteration, 4),
            "elevation_m": round(elevation, 1),
            "slope_deg": round(slope, 1),
            "mn_grade_pct": 0.0,
            "target_occurrence": 0  # Background / Non-mineralized
        })

df_prospectivity = pd.DataFrame(prospectivity_rows)
df_prospectivity.to_csv("manganese_prospectivity_data.csv", index=False)
print(f"Saved manganese_prospectivity_data.csv with {len(df_prospectivity)} records!")

# ==============================================================================
# 3. GENERATE DATASET 2: PRODUCTION & SHORTFALL (For Shaurya's Model 2)
# ==============================================================================
# 36 months (3 years: 2022 to 2024) across all 10 MOIL mines = 360 monthly records
# Incorporates your research: Monsoon impact (July 342mm), stripping backlogs, hoist maintenance
shortfall_rows = []

for mine in MOIL_MINES:
    monthly_target = mine["annual_capacity"] / 12.0
    
    for year in [2022, 2023, 2024]:
        for month in range(1, 13):
            # Monsoon seasonality (June-September: high rain)
            if month in [6, 7, 8, 9]:
                rainfall_mm = np.random.uniform(200, 380) if month == 7 else np.random.uniform(120, 260)
            else:
                rainfall_mm = np.random.uniform(0, 35)
                
            # Summer temperature (April-May: 42-45°C)
            temp_c = np.random.uniform(40, 46) if month in [4, 5] else np.random.uniform(22, 36)
            
            # Equipment uptime & breakdown hours
            uptime_pct = np.random.uniform(72, 92)
            breakdown_hrs = np.random.uniform(12, 65)
            blasting_delays = np.random.poisson(2 if mine['type'] == 'Opencast' else 1)
            
            # Physical shortfall penalty calculation (Anchored in domain reality)
            shortfall_penalty = 0.0
            
            # Monsoon penalty: severe for Opencast (pit flooding at Sitapatore/Dongri Buzurg),
            # moderate for Underground (pumping strain at Balaghat)
            if rainfall_mm > 150:
                shortfall_penalty += (0.18 if mine['type'] == 'Opencast' else 0.08)
                
            # Breakdown penalty
            if breakdown_hrs > 40:
                shortfall_penalty += 0.12
            if uptime_pct < 78:
                shortfall_penalty += 0.08
            
            # Random operational variance (+/- 4%)
            noise = np.random.normal(0, 0.04)
            actual_shortfall_pct = np.clip(shortfall_penalty + noise, -0.05, 0.40)
            
            actual_production = round(monthly_target * (1.0 - actual_shortfall_pct))
            shortfall_tonnes = round(monthly_target - actual_production)
            
            shortfall_rows.append({
                "mine_id": mine["mine_id"],
                "mine_name": mine["name"],
                "mine_type": mine["type"],
                "year": year,
                "month": month,
                "monthly_target_tonnes": round(monthly_target),
                "actual_production_tonnes": actual_production,
                "shortfall_tonnes": max(0, shortfall_tonnes),
                "shortfall_pct": round(max(0, actual_shortfall_pct * 100), 2),
                "rainfall_mm": round(rainfall_mm, 1),
                "avg_temperature_c": round(temp_c, 1),
                "equipment_uptime_pct": round(uptime_pct, 1),
                "breakdown_hours": round(breakdown_hrs, 1),
                "blasting_delays_count": blasting_delays,
                "avg_feed_grade_pct": mine["avg_grade_mn"]
            })

df_shortfall = pd.DataFrame(shortfall_rows)
df_shortfall.to_csv("production_shortfall_data.csv", index=False)
print(f"Saved production_shortfall_data.csv with {len(df_shortfall)} records!")

Saved moil_mines_master.json!
Saved manganese_prospectivity_data.csv with 487 records!
Saved production_shortfall_data.csv with 360 records!
